In [6]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (20, 12),
        'axes.labelsize': 'medium',
        'axes.titlesize':'medium',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import time
import datetime
import pytz

NYC_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.Unified.registry import UnifiedValue
from Query.Unified.UnifiedQuery import UnifiedQuery
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder

In [8]:
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
ts_builder = TimeseriesBuilder()

In [9]:
start = NYC_tz.localize(datetime.datetime(2026, 2, 20, 18, 00))
end = NYC_tz.localize(datetime.datetime(2026, 3, 20, 17, 00))

q = UnifiedQuery(
    curve="USD-SOFR-1D-Q12STIRT",
    tenor="IMM_12xIMM_13",
    value=UnifiedValue.IRS_RATE,
)
df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[q],
    freq="1min",
    n_jobs=12,
    routers={
        "IRS": IRSwapsTB(curve_mdp, show_tqdm=True),
    },
)
df

WARNING	Task(Task-3) Caching.computed_timeseries_store:computed_timeseries_store.py:_open_duckdb_graceful()- DuckDB unavailable (file locked), falling back to parquet-only: C:\Users\chris\clee\ARBS\data\ts\computed_ts.duckdb


READING USD-SOFR-1D-Q12STIRT computed cache...:   0%|          | 0/1 [00:00<?, ?it/s]

,USD-SOFR-1D IMM_12xIMM_13 OUTRIGHT RATE
Date,
2026-02-22 18:00:00-05:00,3.267223
2026-02-22 18:01:00-05:00,3.268131
2026-02-22 18:02:00-05:00,3.268142
2026-02-22 18:03:00-05:00,3.268142
2026-02-22 18:04:00-05:00,3.268142
...,...
2026-03-20 16:56:00-04:00,3.541681
2026-03-20 16:57:00-04:00,3.541681
2026-03-20 16:58:00-04:00,3.541681


In [20]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(
    df[q.col_name().replace("-Q12STIRT", "")],
    which="left",
    indicators=[
        # {"kind": "last", "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},  
        # {"kind": "sma", "window": 60, "style": {"linestyle": "--", "color": "green"}},
    ],
    # ou={"enable": True, "steps": 126, "add_metrics_to_legend": True}
)
legend(show_date=True, loc="upper left")
plt.show()